In [33]:
import tensorflow as tf
import os, datetime
import tensorflow_recommenders as tfrs
from google.cloud import bigquery
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import pprint
from typing import Dict, Text
import logging
import faiss
from keras_tuner import HyperModel, HyperParameters
from keras_tuner.tuners import BayesianOptimization, Hyperband, RandomSearch
from keras_tuner import Objective

import xgboost as xgb
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
# Check for GPU availability and set memory growth
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logger.info(f"GPUs available: {len(gpus)}")
    except RuntimeError as e:
        logger.warning(e)
else:
    logger.info("No GPUs available.")

2025-01-22 20:33:10,902 - INFO - No GPUs available.


In [4]:
# Define the BigQuery table and project details
PROJECT_ID = 'oolola'
DATASET_ID = 'movie_data'
TABLE_ID   = 'preprocessed_data'
timestamp  = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
output_dir = 'gs://movie-data-1/trained-model'
# Function to load movies from BigQuery
def load_movies_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT title, genres
        FROM `{PROJECT_ID}.{DATASET_ID}.preprocessed_movies`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading movies from BigQuery: {e}")
        raise
# Function to load ratings from BigQuery
def load_ratings_bq():
    try:
        client = bigquery.Client(project=PROJECT_ID)
        query = f"""
        SELECT user_id, title, rating
        FROM `{PROJECT_ID}.{DATASET_ID}.ratings_with_titles`
        """
        query_job = client.query(query)
        return query_job.to_dataframe()
    except Exception as e:
        logger.error(f"Error loading ratings from BigQuery: {e}")
        raise

# Function to get movie titles based on movie IDs
def get_titles(movie_ids: list):
    try:
        client = bigquery.Client(project=PROJECT_ID)
        # Convert the list of movie IDs to a comma-separated string
        movie_ids_str = ', '.join(map(str, movie_ids))
        query = f"""
        SELECT title
        FROM `{PROJECT_ID}.{DATASET_ID}.genres`
        WHERE movie_id IN ({movie_ids_str})
        """
        query_job = client.query(query)
        return query_job.to_dataframe()['title'].tolist()
    except Exception as e:
        logger.error(f"Error getting titles from BigQuery: {e}")
        raise

In [5]:
movies_bq = load_movies_bq()
movies_dict = {key: list(value) for key, value in movies_bq[['title', 'genres']].to_dict(orient='list').items()}

In [6]:
ratings_bq = load_ratings_bq()
ratings_dict = {key: list(value) for key, value in ratings_bq[['title', 'user_id', 'rating']].to_dict(orient='list').items()}

In [7]:
ratings = tf.data.Dataset.from_tensor_slices(ratings_dict)

2025-01-22 20:33:23.269750: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [8]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'rating': 2.0, 'title': b'Initiation', 'user_id': 56703}


In [9]:
movies = tf.data.Dataset.from_tensor_slices(movies_dict)

In [10]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]),
 'title': b'Siberian Sniper'}


In [11]:
ratings = ratings.map(lambda x: {
            "title": x["title"],
            "user_id": x["user_id"],
            "rating": x["rating"]
        })
movies = movies.map(lambda x: {
    "title": x["title"],
    "genres": x["genres"]
})

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


2025-01-22 20:33:24,597 - WARNING - From /Users/dmmckinn/repos/movie_recommender/.venv/lib/python3.10/site-packages/tensorflow/python/autograph/pyct/static_analysis/liveness.py:83: Analyzer.lamba_check (from tensorflow.python.autograph.pyct.static_analysis.liveness) is deprecated and will be removed after 2023-09-23.
Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089


In [12]:
# Create a dictionary from the movies dataset
movies_dict = {movie["title"].numpy().decode('utf-8'): movie["genres"].numpy() for movie in movies}

def combine_datasets(rating):
    def lookup_genres(title):
        title_str = title.numpy().decode('utf-8')  # Convert to numpy and decode the title from bytes to string
        return movies_dict.get(title_str, [0] * 19)

    genres = tf.py_function(
        func=lookup_genres,
        inp=[rating["title"]],
        Tout=tf.int32
    )
    genres.set_shape([19])
    rating["genres"] = genres
    return rating

combined_dataset = ratings.map(combine_datasets)

In [13]:
# print one example of combined dataset
for x in combined_dataset.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'genres': array([0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0],
      dtype=int32),
 'rating': 2.0,
 'title': b'Initiation',
 'user_id': 56703}


In [14]:
combined_dataset = combined_dataset.map(lambda x: {
    "title": x["title"],
    "user_id": x["user_id"],
    "genres": x["genres"],
    "rating": x["rating"]
}, num_parallel_calls=tf.data.AUTOTUNE)

In [15]:
# Set the seed for reproducibility
tf.random.set_seed(42)

# Shuffle the dataset with a large buffer size
# Ensure the buffer size is large enough to cover randomness but not too large to exhaust memory
shuffle_buffer_size = 200000  # Smaller buffer size for faster shuffling
shuffled = combined_dataset.shuffle(buffer_size=shuffle_buffer_size, seed=42, reshuffle_each_iteration=True)

# Calculate relative proportions for splitting
train_ratio = 0.8

ds_length = int(tf.data.experimental.cardinality(shuffled).numpy())
print(f"Length of the dataset: {ds_length}")

# Define the split function for large datasets
def split_dataset(dataset, train_ratio):
    # Determine split points
    trainds = dataset.take(int(train_ratio * ds_length))
    testds = dataset.skip(int(train_ratio * ds_length))

    return trainds, testds

# Perform the split
trainds, testds = split_dataset(shuffled, train_ratio)

# Optimize performance with prefetching
train_batch_size =  128
eval_test_batch_size = 64

# Optimize datasets with batching, caching, and prefetching
trainds = trainds.batch(train_batch_size).cache().prefetch(tf.data.AUTOTUNE)
testds = testds.batch(eval_test_batch_size).cache().prefetch(tf.data.AUTOTUNE)

Length of the dataset: 174490


In [16]:
titles = movies.batch(100000).map(lambda x: x["title"])
userIds = ratings.batch(1000000).map(lambda x: x["user_id"])
genres = movies.batch(100000).map(lambda x: {
        "title": x["title"],
        "genres": x["genres"]
    })

In [17]:
unique_titles = np.unique(np.concatenate(list(titles)))
unique_userIds = np.unique(np.concatenate(list(userIds)))
unique_genres = [
            'Action',
            'Adventure',
            'Animation',
            'Children',
            'Comedy',
            'Crime',
            'Drama',
            'Documentary',
            'Fantasy',
            'Film-Noir',
            'Horror',
            'IMAX',
            'Musical',
            'Mystery',
            'Romance',
            'Sci-Fi',
            'Thriller',
            'War',
            'Western'
        ]

In [18]:
class RecommendationModel(tfrs.Model):
    def __init__(self, user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight=1.0, retrieval_weight=1.0):
        super().__init__()
        self.user_model = user_model
        self.movie_model = movie_model
        self.genre_model = genre_model
        self.rating_model = rating_model
        self.rating_task = rating_task
        self.retrieval_task = retrieval_task
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def call(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        user_embeddings = self.user_model(features["user_id"])
        movie_embeddings = self.movie_model(features["title"])
        genre_embeddings = self.genre_model(features["genres"])
        rating_predictions = self.rating_model([features["user_id"], features["title"], features["genres"]])
        return user_embeddings, movie_embeddings, genre_embeddings, rating_predictions

    def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
        ratings = features.pop("rating")
        user_embeddings, movie_embeddings, genre_embeddings, rating_predictions = self(features)
        rating_loss = self.rating_task(labels=ratings, predictions=rating_predictions)
        retrieval_loss = self.retrieval_task(user_embeddings, movie_embeddings)
        return (self.rating_weight * rating_loss
                + self.retrieval_weight * retrieval_loss)

In [19]:
# # Function to create the model
# def create_model(unique_user_ids, unique_movie_ids, num_genres, rating_weight=1.0, retrieval_weight=1.0):
#     embedding_dimension = 128
#     user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
#     movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
#     genre_input = tf.keras.layers.Input(shape=(num_genres,), dtype=tf.float32, name='genres')

#     user_lookup = tf.keras.layers.IntegerLookup(vocabulary=unique_user_ids, mask_token=None)
#     movie_lookup = tf.keras.layers.StringLookup(vocabulary=unique_titles, mask_token=None)

#     user_embedding = tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
#     movie_embedding = tf.keras.layers.Embedding(len(unique_movie_ids) + 1, embedding_dimension)(movie_lookup(movie_input))
#     genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

#     concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

#     dense_1 = tf.keras.layers.Dense(256, activation="relu")(concatenated_embeddings)
#     dropout_1 = tf.keras.layers.Dropout(0.5)(dense_1)
#     dense_2 = tf.keras.layers.Dense(128, activation="relu")(dropout_1)
#     dropout_2 = tf.keras.layers.Dropout(0.5)(dense_2)
#     rating_output = tf.keras.layers.Dense(1)(dropout_2)


#     user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
#     movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
#     genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
#     rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

#     metrics = tfrs.metrics.FactorizedTopK(
#         candidates=tf.data.Dataset.from_tensor_slices(unique_movie_ids).batch(128).map(movie_model)
#     )
#     rating_task = tfrs.tasks.Ranking(
#         loss=tf.keras.losses.MeanSquaredError(),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()],
#     )
#     retrieval_task = tfrs.tasks.Retrieval(
#         metrics=metrics
#     )

#     model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, rating_weight, retrieval_weight)
#     lr_schedule_ed = tf.keras.optimizers.schedules.ExponentialDecay(
#         initial_learning_rate=0.1,
#         decay_steps=545,
#         decay_rate=0.9,
#         staircase=True
#     )

#     # lr_schedule_pcd = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
#     #     boundaries=[2200, 3000, 3500],
#     #     values=[0.1, 0.01, 0.001, 0.0001]
#     # )
    
#     # Use the learning rate schedule in the optimizer
#     model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=lr_schedule_ed))
#     return model


In [20]:
class RecommendationHyperModel(HyperModel):
    def __init__(self, unique_user_ids, unique_titles, num_genres, rating_weight=1.0, retrieval_weight=1.0):
        self.unique_user_ids = unique_user_ids
        self.unique_titles = unique_titles
        self.num_genres = num_genres
        self.rating_weight = rating_weight
        self.retrieval_weight = retrieval_weight

    def build(self, hp):
        embedding_dimension = hp.Int('embedding_dimension', min_value=32, max_value=256, step=32)
        user_input = tf.keras.layers.Input(shape=(), dtype=tf.int32, name='user_id')
        movie_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='title')
        genre_input = tf.keras.layers.Input(shape=(self.num_genres,), dtype=tf.float32, name='genres')

        user_lookup = tf.keras.layers.IntegerLookup(vocabulary=self.unique_user_ids, mask_token=None)
        movie_lookup = tf.keras.layers.StringLookup(vocabulary=self.unique_titles, mask_token=None)

        user_embedding = tf.keras.layers.Embedding(len(self.unique_user_ids) + 1, embedding_dimension)(user_lookup(user_input))
        movie_embedding = tf.keras.layers.Embedding(len(self.unique_titles) + 1, embedding_dimension)(movie_lookup(movie_input))
        genre_embedding = tf.keras.layers.Dense(embedding_dimension)(genre_input)

        concatenated_embeddings = tf.concat([user_embedding, movie_embedding, genre_embedding], axis=1)

        dense_1 = tf.keras.layers.Dense(hp.Int('units_1', min_value=128, max_value=512, step=64), activation="relu")(concatenated_embeddings)
        dropout_1 = tf.keras.layers.Dropout(hp.Float('dropout_1', min_value=0.1, max_value=0.5, step=0.1))(dense_1)
        dense_2 = tf.keras.layers.Dense(hp.Int('units_2', min_value=64, max_value=256, step=32), activation="relu")(dropout_1)
        dropout_2 = tf.keras.layers.Dropout(hp.Float('dropout_2', min_value=0.1, max_value=0.5, step=0.1))(dense_2)
        rating_output = tf.keras.layers.Dense(1)(dropout_2)

        user_model = tf.keras.Model(inputs=user_input, outputs=user_embedding)
        movie_model = tf.keras.Model(inputs=movie_input, outputs=movie_embedding)
        genre_model = tf.keras.Model(inputs=genre_input, outputs=genre_embedding)
        rating_model = tf.keras.Model(inputs=[user_input, movie_input, genre_input], outputs=rating_output)

        metrics = tfrs.metrics.FactorizedTopK(
            candidates=tf.data.Dataset.from_tensor_slices(self.unique_titles).batch(128).map(movie_model)
        )
        rating_task = tfrs.tasks.Ranking(
            loss=tf.keras.losses.MeanSquaredError(),
            metrics=[tf.keras.metrics.RootMeanSquaredError()],
        )
        retrieval_task = tfrs.tasks.Retrieval(
            metrics=metrics
        )
        model = RecommendationModel(user_model, movie_model, genre_model, rating_model, rating_task, retrieval_task, self.rating_weight, self.retrieval_weight)
        
        # Define the learning rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log'),
            decay_steps=hp.Int('decay_steps', min_value=500, max_value=1000, step=100),
            decay_rate=hp.Float('decay_rate', min_value=0.8, max_value=0.9, step=0.05),
            staircase=True
        )

        # # Define the learning rate schedule using CosineDecay
        # lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        #     initial_learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-1, sampling='log'),
        #     decay_steps=1000,
        #     alpha=0.1
        # )
        
        # Use the learning rate schedule in the optimizer
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule))
        return model

In [21]:
tuner = Hyperband(
    RecommendationHyperModel(unique_userIds, unique_titles, len(unique_genres)),
    objective=Objective("val_factorized_top_k/top_5_categorical_accuracy", direction="max"),
    max_epochs=12,
    # max_trials=10,
    factor=3,
    # executions_per_trial=1,
    directory='tuning/tpe',
    project_name='20250122063906/movie_recommendation',
)

# Reload the tuner to load previous results
# tuner.reload()

tuner.search(trainds, epochs=12, validation_data=testds, callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)])

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
tuned_model = tuner.hypermodel.build(best_hps)

Reloading Tuner from tuning/tpe/20250122063906/movie_recommendation/tuner0.json


In [22]:
best_hps.values

{'embedding_dimension': 224,
 'units_1': 320,
 'dropout_1': 0.4,
 'units_2': 224,
 'dropout_2': 0.4,
 'learning_rate': 0.0022820322663096213,
 'decay_steps': 1000,
 'decay_rate': 0.8500000000000001,
 'tuner/epochs': 12,
 'tuner/initial_epoch': 4,
 'tuner/bracket': 2,
 'tuner/round': 2,
 'tuner/trial_id': '0013'}

In [54]:
def generate_unique_genres(n):
    """
    Generate a one-hot encoded array for `n` unique genres.
    
    Args:
        n (int): Number of unique genres.
    
    Returns:
        np.ndarray: A one-hot encoded array of shape (n, n).
    """
    return np.eye(n, dtype=int).tolist()
    
unique_genres_array = generate_unique_genres(len(unique_genres))

# Extract user and movie embeddings
user_embeddings = tuned_model.user_model.predict(unique_userIds)
movie_embeddings = tuned_model.movie_model.predict(unique_titles)
genre_embeddings = tuned_model.genre_model.predict(unique_genres_array)


# Create a dictionary to map user IDs and movie IDs to their embeddings
user_embedding_dict = {user_id: embedding for user_id, embedding in zip(unique_userIds, user_embeddings)}
movie_embedding_dict = {movie_id: embedding for movie_id, embedding in zip(unique_titles, movie_embeddings)}
genre_embedding_dict = {i: embedding for i, embedding in enumerate(genre_embeddings)}

user_embedding_dict = {k: v for k, v in user_embedding_dict.items()}
movie_embedding_dict = {k.decode('utf-8'): v for k, v in movie_embedding_dict.items()}
genre_embedding_dict = {k: v for k, v in genre_embedding_dict.items()}

1/1 [==============================] - 0s 41ms/step


In [57]:
# Convert ratings to pandas DataFrame
ratings_df = pd.DataFrame(combined_dataset.as_numpy_iterator())
print(ratings_df.head())

                          title  user_id  \
0                 b'Initiation'    56703   
1              b'Anything Goes'   125172   
2  b'1000 Miles From Christmas'   129858   
3                   b'Eternals'   174766   
4                   b'Eternals'    55681   

                                              genres  rating  
0  [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, ...     2.0  
1  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     2.0  
2  [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...     2.0  
3  [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     2.0  
4  [1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...     2.0  


In [60]:
# Create feature vectors by combining user and movie embeddings
def create_feature_vector(row):
    user_id = row['user_id']
    title = row['title'].decode('utf-8')
    genres = row['genres']
    
    if user_id not in user_embedding_dict:
        raise KeyError(f"User ID {user_id} not found in user_embedding_dict")
    if title not in movie_embedding_dict:
        raise KeyError(f"Title '{title}' not found in movie_embedding_dict")
    
    user_embedding = user_embedding_dict[user_id]
    movie_embedding = movie_embedding_dict[title]
    
    # Calculate the genre embedding by multiplying the one-hot vector with the genre embeddings
    genre_embedding = np.dot(genres, np.array([genre_embedding_dict[i] for i in range(len(genres))]))
    
    return np.concatenate([user_embedding, movie_embedding, genre_embedding])

# Apply the function to create feature vectors
try:
    ratings_df['features'] = ratings_df.apply(lambda row: create_feature_vector(row), axis=1)
except KeyError as e:
    print(e)

# Set the seed for reproducibility
random_seed = 42

# Shuffle the DataFrame
shuffled_df = ratings_df.sample(frac=1, random_state=random_seed).reset_index(drop=True)
train_df, val_df = train_test_split(shuffled_df, test_size=0.2, random_state=random_seed)

X_train = np.vstack(train_df['features'].values)
y_train = train_df['rating'].values
X_val = np.vstack(val_df['features'].values)
y_val = val_df['rating'].values

# Group sizes for ranking
group_train = train_df.groupby('user_id').size().to_list()
group_val = val_df.groupby('user_id').size().to_list()

# Create DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtrain.set_group(group_train)
dval = xgb.DMatrix(X_val, label=y_val)
dval.set_group(group_val)

In [61]:
# Define parameters for XGBoost
params = {
    'objective': 'rank:pairwise',  # Pairwise ranking objective
    'eval_metric': 'ndcg',  # Normalized Discounted Cumulative Gain
    'eta': 0.1,
    'max_depth': 6
}

# Train the XGBoost model
bst = xgb.train(params, dtrain, num_boost_round=100, evals=[(dval, 'eval')], early_stopping_rounds=10)

[0]	eval-ndcg:0.94219
[1]	eval-ndcg:0.94354
[2]	eval-ndcg:0.94325
[3]	eval-ndcg:0.94349
[4]	eval-ndcg:0.94356
[5]	eval-ndcg:0.94388
[6]	eval-ndcg:0.94418
[7]	eval-ndcg:0.94430
[8]	eval-ndcg:0.94415
[9]	eval-ndcg:0.94437
[10]	eval-ndcg:0.94470
[11]	eval-ndcg:0.94468
[12]	eval-ndcg:0.94473
[13]	eval-ndcg:0.94475
[14]	eval-ndcg:0.94475
[15]	eval-ndcg:0.94476
[16]	eval-ndcg:0.94476
[17]	eval-ndcg:0.94485
[18]	eval-ndcg:0.94487
[19]	eval-ndcg:0.94474
[20]	eval-ndcg:0.94476
[21]	eval-ndcg:0.94492
[22]	eval-ndcg:0.94487
[23]	eval-ndcg:0.94492
[24]	eval-ndcg:0.94496
[25]	eval-ndcg:0.94535
[26]	eval-ndcg:0.94528
[27]	eval-ndcg:0.94570
[28]	eval-ndcg:0.94589
[29]	eval-ndcg:0.94592
[30]	eval-ndcg:0.94594
[31]	eval-ndcg:0.94601
[32]	eval-ndcg:0.94616
[33]	eval-ndcg:0.94631
[34]	eval-ndcg:0.94637
[35]	eval-ndcg:0.94659
[36]	eval-ndcg:0.94706
[37]	eval-ndcg:0.94718
[38]	eval-ndcg:0.94732
[39]	eval-ndcg:0.94764
[40]	eval-ndcg:0.94778
[41]	eval-ndcg:0.94792
[42]	eval-ndcg:0.94810
[43]	eval-ndcg:0.9485

In [68]:
# Predict ratings for the validation set
y_pred = bst.predict(dval)

# # Evaluate the model
# rmse = np.sqrt(np.mean((y_val - y_pred) ** 2))
# print(f'Validation RMSE: {rmse}')

# Assuming val_df contains the validation data with columns: user_id, title, rating
val_df['predicted_rating'] = y_pred

# Rank items for each user
val_df['rank'] = val_df.groupby('user_id')['predicted_rating'].rank(ascending=False, method='first')
def top_k_accuracy(df, k):
    # Check if the true item is among the top K recommended items
    df['correct'] = df['rank'] <= k
    # Calculate the top-K accuracy
    top_k_acc = df.groupby('user_id')['correct'].max().mean()
    return top_k_acc

# Calculate top-10 accuracy
top_10_accuracy = top_k_accuracy(val_df, k=10)
print(f'Top-10 Accuracy: {top_10_accuracy}')

# Rank items for each user
val_df['rank'] = val_df.groupby('user_id')['predicted_rating'].rank(ascending=False, method='first')

# Use the model for ranking
def rank_movies(user_id, unique_titles, num_movies=5):
    user_embedding = user_embedding_dict[user_id]
    candidate_features = [np.concatenate([user_embedding, movie_embedding_dict[movie_id]]) for movie_id in unique_titles]
    dtest = xgb.DMatrix(np.vstack(candidate_features))
    predicted_ratings = bst.predict(dtest)
    ranked_movies = [movie_id for _, movie_id in sorted(zip(predicted_ratings, unique_titles), reverse=True)]
    return ranked_movies

# Example usage
user_id = 56703
candidate_movie_ids = [1, 2, 3, 4, 5]
unique_titles_nb = [title.decode('utf-8') for title in unique_titles]

ranked_movies = rank_movies(user_id, unique_titles_nb)
print(f'Ranked movies for user {user_id}: {ranked_movies}')

Top-10 Accuracy: 1.0
Ranked movies for user 56703: ['Spider-Man: Across the Spider-Verse', 'How to Blow Up a Pipeline', 'Prisoners of the Ghostland', 'The Last Kingdom: Seven Kings Must Die', 'The Phenomenon', 'The Wedding Veil Unveiled', "Don't Look Up", 'Say Hey, Willie Mays!', 'Back to the Outback', 'Orca', 'Vivo', 'Pompo: The Cinéphile', 'A Boy Called Christmas', 'Togo', 'Psych 3: This Is Gus', 'Maneater', 'Vengeance is Mine', 'OLIVIA RODRIGO: driving home 2 u (a SOUR film)', 'Amityville Christmas Vacation', 'Squared Love', 'Squaring the Circle (The Story of Hipgnosis)', 'Wrong Place', 'A Man Called Otto', 'The Point Men', 'Yamabuki', 'The Tasting', 'Unicorn Town', 'Persona Non Grata', 'The Bright Side', 'Lost Bullet 2', 'Egregor', 'Minions & More 1', 'The Siege', 'I Am Zlatan', "Satan's Slaves 2: Communion", 'Arranged Love', 'Unwelcome', 'Kala', 'Thank You', 'Terry Bradshaw: Going Deep', 'Found', 'Hero Dog: The Journey Home', 'Klokkenluider', 'Adele One Night Only', 'Wicked (The W

In [48]:
# Define callbacks for training
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
model_export_path = os.path.join(output_dir, 'saved-model', timestamp)
checkpoint_path = os.path.join(output_dir, 'checkpoints', f"{timestamp}_cp.ckpt")
tensorboard_path = os.path.join(output_dir, 'tensorboard', timestamp)
faiss_path = os.path.join(output_dir, 'faiss', f"{timestamp}_faiss.index")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"trained_model/tpe/checkpoints/{timestamp}_cp.ckpt",
    save_best_only=True,
    save_weights_only=True,
    monitor='factorized_top_k/top_5_categorical_accuracy',
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(f"trained_model/tpe/tensorboard/{timestamp}_cp.ckpt", histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='factorized_top_k/top_10_categorical_accuracy',
    patience=2,
    restore_best_weights=True
)

In [31]:
model = create_model(unique_userIds, unique_titles, len(unique_genres), rating_weight=1.0, retrieval_weight=1.0)

In [ ]:
# Define callbacks for training
timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
model_export_path = os.path.join(output_dir, 'saved-model', timestamp)
checkpoint_path = os.path.join(output_dir, 'checkpoints', f"{timestamp}_cp.ckpt")
tensorboard_path = os.path.join(output_dir, 'tensorboard', timestamp)
faiss_path = os.path.join(output_dir, 'faiss', f"{timestamp}_faiss.index")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"trained_model/bayesian/checkpoints/{timestamp}_cp.ckpt",
    save_best_only=True,
    save_weights_only=True,
    monitor='factorized_top_k/top_5_categorical_accuracy',
    mode='max'
)
tensorboard_callback = tf.keras.callbacks.TensorBoard(f"trained_model/bayesian/tensorboard/{timestamp}_cp.ckpt", histogram_freq=1)
early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='factorized_top_k/top_10_categorical_accuracy',
    patience=2,
    restore_best_weights=True
)

bayesian_tuner = BayesianOptimization(
    RecommendationHyperModel(unique_userIds, unique_titles, len(unique_genres)),
    objective=Objective("val_factorized_top_k/top_5_categorical_accuracy", direction="max"),
    max_trials=10,
    executions_per_trial=1,
    directory='tuning/bayesian_optimization',
    project_name='20250120133706/movie_recommendation',
)
bayesian_tuner.reload()
# Get the best hyperparameters
best_hps = bayesian_tuner.get_best_hyperparameters(num_trials=1)[0]

# Build the model with the best hyperparameters
bayesian_model = bayesian_tuner.hypermodel.build(best_hps)
# Build the model by calling it on a batch of data
bayesian_model.fit(
    trainds,
    epochs=10,
    # validation_data=testds,
    callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

In [ ]:
# Build the model by calling it on a batch of data
model.build(input_shape={
	"user_id": (None,),
	"title": (None,),
	"genres": (None, len(unique_genres))
})

In [ ]:
# Train the model
history = tuned_model.fit(
    trainds,
    epochs=12,
    # steps_per_epoch=5,
    # validation_data=testds,
    callbacks=[checkpoint_callback, tensorboard_callback, early_stopping_callback]
)

In [ ]:
steps = ds_length * 0.2 // eval_test_batch_size
metrics = bayesian_model.evaluate(testds, steps=steps, return_dict=True)

In [ ]:
print(f"Retrieval top-1 accuracy: {metrics['factorized_top_k/top_1_categorical_accuracy']:.3f}.")
print(f"Retrieval top-5 accuracy: {metrics['factorized_top_k/top_5_categorical_accuracy']:.3f}.")
print(f"Retrieval top-10 accuracy: {metrics['factorized_top_k/top_10_categorical_accuracy']:.3f}.")
print(f"Ranking RMSE: {metrics['root_mean_squared_error']:.3f}.")

In [53]:
tuned_model.save_weights(f"trained_model/tpe/weights/{timestamp}_weights.h5")

In [ ]:
trained_user_embeddings, trained_movie_embeddings, trained_genre_embeddings, predicted_rating = tuned_model({
      "user_id": np.array([56703]),
      "title": np.array(['Siberian Sniper']),
      "genres": np.array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]])
  })
print("Predicted rating:")
print(predicted_rating)

In [52]:
def extract_embeddings(model, unique_user_ids, unique_titles):
    """
    Extract embeddings for all users and movies using the trained model.
    
    Args:
    - model: The trained recommendation model.
    - unique_user_ids: List of unique user IDs.
    - unique_movie_ids: List of unique movie IDs.
    
    Returns:
    - user_embeddings: Embeddings for all users.
    - movie_embeddings: Embeddings for all movies.
    """
    # Extract movie embeddings
    titles = np.array(unique_titles)
    movie_embeddings = model.movie_model(tf.constant(titles, dtype=tf.string)).numpy()

    # Extract user embeddings
    user_ids = np.array(unique_user_ids)
    user_embeddings = model.user_model(tf.constant(user_ids, dtype=tf.int32)).numpy()

    return user_embeddings, movie_embeddings

In [53]:
def index_movie_embeddings(movie_embeddings):
    """
    Index the movie embeddings using FAISS.
    
    Args:
    - movie_embeddings: Embeddings for all movies.
    
    Returns:
    - index: FAISS index with movie embeddings.
    """
    # Dimension of the embeddings
    embedding_dimension = movie_embeddings.shape[1]

    # Create a FAISS index
    index = faiss.IndexFlatL2(embedding_dimension)

    # Add movie embeddings to the index
    index.add(movie_embeddings)

    return index

In [54]:
def recommend_movies(model, index, unique_titles, user_id, k=10):
    """
    Recommend movies for a given user by querying the FAISS index.
    
    Args:
    - model: The trained recommendation model.
    - index: FAISS index with movie embeddings.
    - unique_titles: List of unique movie titles.
    - user_id: The user ID for which to make recommendations.
    - k: Number of recommendations to retrieve (default is 10).
    
    Returns:
    - recommended_movie_ids: List of recommended movie IDs.
    """
    # Get the embedding for the given user
    user_embedding = model.user_model(tf.constant([user_id], dtype=tf.int32)).numpy()

    # Query the FAISS index
    distances, indices = index.search(user_embedding, k)

    # Get the recommended movie IDs
    recommended_movies = np.array(unique_titles)[indices[0]]

    return recommended_movies

In [ ]:
# Extract embeddings
user_embeddings, movie_embeddings = extract_embeddings(tuned_model, unique_userIds, unique_titles)

# Index movie embeddings
index = index_movie_embeddings(movie_embeddings)

# Example user ID for which to make recommendations
example_user_id = 56703

recommended_movie_ids = recommend_movies(tuned_model, index, unique_titles, example_user_id, k=3)

print(f"Recommended movie titles for user {example_user_id}: {recommended_movie_ids}")

In [61]:
faiss.write_index(index, "trained_model/faiss/{}_faiss.index".format(timestamp))